# 02 · 모델 학습 — 여러 정상 파일 합침 (Multi-file Training)

정상(train) 파일 **리스트**를 세로로 이어붙여 하나의 큰 정상 데이터로 만든 뒤 학습한다.
- 리스트에 파일 1개 → **단일 학습**
- 리스트에 여러 개 → **합침 학습** (정상 패턴 다양성 ↑ → 오탐 ↓, 일반화 ↑)

**충전(chg)과 방전(dchg)은 패턴이 정반대이므로 절대 섞지 않는다.**
`SIGNAL_TYPE`으로 한 종류만 골라 학습하고, 모델은 종류별 폴더에 저장한다.

**필수**: 여러 파일을 합치려면 **하나의 PCA로 전체를 fit** 해야 하므로 `pca_mode='consistent'`.

In [ ]:
from function_def import *
from parameter import make_config
import os, numpy as np, pandas as pd, joblib
import tensorflow as tf

## 설정 — 어떤 정상 파일들을 합쳐 학습할지

`CELL_IDS`에 넣은 셀 번호들의 정상 파일을 합친다. 예: 단일=[1000], 합침=[1000,1001,1002].
`SIGNAL_TYPE`은 'chg' 또는 'dchg' (섞지 말 것).

In [ ]:
SIGNAL_TYPE = 'chg'                 # 'chg' 또는 'dchg'
CELL_IDS    = [1000, 1001, 1002]    # 합칠 정상 셀 번호. 단일학습은 [1000] 처럼 하나만.
TRAIN_DIR   = './data/preprocessed/train'

# 파일 경로 목록 생성
train_files = [os.path.join(TRAIN_DIR, '%d_%s.csv' % (cid, SIGNAL_TYPE)) for cid in CELL_IDS]
print("합칠 파일 수:", len(train_files))
for f in train_files:
    print("  ", f, "(있음)" if os.path.isfile(f) else "(없음!)")

## 하이퍼파라미터 (config) — 합침 학습은 정합 모드 강제

여러 파일을 하나의 PCA로 묶어야 하므로 pca_mode='consistent' 로 고정.
체크포인트/PCA 저장 경로는 **신호 종류별로 분리**(chg 모델과 dchg 모델이 안 섞이게).

In [ ]:
CFG = make_config(pca_mode='consistent')   # 필요시 win_size=30 등 여기서 조정

# 신호 종류 + 합침 개수로 실험 태그를 만들어 폴더 분리
tag = '%s_%dfiles' % (SIGNAL_TYPE, len(CELL_IDS))
ckpt_dir = os.path.join('checkpoints', tag)
os.makedirs(ckpt_dir, exist_ok=True)
CFG['ckpt_dir']   = ckpt_dir
CFG['pca_path']   = os.path.join(ckpt_dir, 'pca.joblib')
CFG['scaler_path']= os.path.join(ckpt_dir, 'scaler.joblib')

win_size, features_dim, feat_dim = CFG['win_size'], CFG['features_dim'], CFG['feat_dim']
latent_dim, batch_size, n_critic = CFG['latent_dim'], CFG['batch_size'], CFG['n_critic']
epochs, learning_rate, k_size    = CFG['epochs'], CFG['learning_rate'], CFG['k_size']
lstm_units, drop_gen             = CFG['lstm_units'], CFG['dropout_rate_gen']
crit_filters, crit_drop          = CFG['critic_filters'], CFG['critic_dropout']
diffs_n, lags_n, smooth_n        = CFG['diffs_n'], CFG['lags_n'], CFG['smooth_n']
shape                   = CFG['shape']
encoder_input_shape     = CFG['encoder_input_shape']
encoder_reshape_shape   = CFG['encoder_reshape_shape']
generator_input_shape   = CFG['generator_input_shape']
generator_reshape_shape = CFG['generator_reshape_shape']
critic_x_input_shape    = CFG['critic_x_input_shape']
critic_z_input_shape    = CFG['critic_z_input_shape']
print("실험 태그:", tag, "| ckpt_dir:", ckpt_dir)
print("win_size=%d k_size=%d epochs=%d" % (win_size, k_size, epochs))

## 1) 여러 파일 로드 → 세로 concat → 하나의 PCA로 fit

각 파일을 featurize 후 이어붙여 **하나의 정상 데이터셋**을 만든다.
그 전체에 PCA를 한 번만 fit → 모든 정상 파일이 **같은 좌표계**를 공유(합침의 전제).

In [ ]:
# 1-1. 각 파일 featurize 후 concat
frames = []
for f in train_files:
    d = pd.read_csv(f)
    d = diff_smooth_df(d, lags_n, diffs_n, smooth_n)
    frames.append(d)
    print("loaded:", os.path.basename(f), "shape:", d.shape)

data_all = pd.concat(frames, axis=0, ignore_index=True)
print("합친 정상 데이터:", data_all.shape)

In [ ]:
# 1-2. 합친 전체에 PCA fit (하나의 좌표계) + 스케일러 fit
pca = PCA(n_components=features_dim)
data = pca.fit_transform(data_all)

df_1 = []
for i in range(len(data)):
    df_1.append([i + 1] + [data[i][jj] for jj in range(features_dim)])
df = pd.DataFrame(df_1)
df.columns = ['date'] + ['pca_%s' % str(i) for i in range(1, features_dim + 1)]

X, index = time_segments_aggregate(df, interval=1, time_column='date')
X = SimpleImputer().fit_transform(X)
scaler = MinMaxScaler(feature_range=(-1, 1))
X = scaler.fit_transform(X)

# 정합 모드: pca/scaler 저장 -> 테스트가 동일 좌표계로 transform
joblib.dump(pca, CFG['pca_path'])
joblib.dump(scaler, CFG['scaler_path'])
print("saved pca/scaler ->", ckpt_dir)

X, y, X_index, y_index = rolling_window_sequences(
    X, index, window_size=win_size, target_size=1, step_size=1, target_column=0)
print("학습 시퀀스:", X.shape)

## 2) 네트워크 생성 + 합성 모델

In [ ]:
encoder   = build_encoder_layer(encoder_input_shape, encoder_reshape_shape,
                                win_size=win_size, latent_dim=latent_dim)
generator = build_generator_layer(generator_input_shape, generator_reshape_shape,
                                  win_size=win_size, features_dim=features_dim,
                                  lstm_units=lstm_units, dropout_rate=drop_gen)
critic_x  = build_critic_x_layer(critic_x_input_shape, k_size=k_size,
                                 filters=crit_filters, dropout_rate=crit_drop)
critic_z  = build_critic_z_layer(critic_z_input_shape)
optimizer = tf.keras.optimizers.Adam(learning_rate)

z = Input(shape=(latent_dim, 1)); x = Input(shape=shape)
x_ = generator(z); z_ = encoder(x)
critic_x_model = Model([x, z], [critic_x(x), critic_x(x_), RandomWeightedAverage(batch_size)([x, x_])])
critic_z_model = Model([x, z], [critic_z(z), critic_z(z_), RandomWeightedAverage(batch_size)([z, z_])])
z_gen = Input(shape=(latent_dim, 1)); x_gen = Input(shape=shape)
x_gen_ = generator(z_gen); z_gen_ = encoder(x_gen); x_gen_rec = generator(z_gen_)
encoder_generator_model = Model([x_gen, z_gen], [critic_x(x_gen_), critic_z(z_gen_), x_gen_rec])
print("models ready")

## 3) 학습 루프

In [ ]:
# 재현용 seed (선택): np.random.seed(42); tf.random.set_seed(42)
X = X.reshape((-1, shape[0], feat_dim))
X_ = np.copy(X)
fake  =  np.ones((batch_size, 1), dtype=np.float32)
valid = -np.ones((batch_size, 1), dtype=np.float32)
delta =  np.ones((batch_size, 1), dtype=np.float32)

for epoch in range(1, epochs + 1):
    np.random.shuffle(X_)
    g_loss, cx_loss, cz_loss = [], [], []
    mb_size = batch_size * n_critic
    for i in range(int(X_.shape[0] // mb_size)):
        mb = X_[i * mb_size:(i + 1) * mb_size]
        critic_x.trainable = True;  critic_z.trainable = True
        generator.trainable = False; encoder.trainable = False
        for j in range(n_critic):
            xb = mb[j * batch_size:(j + 1) * batch_size]
            zb = np.random.normal(size=(batch_size, latent_dim, 1))
            cx_loss.append(critic_x_train_on_batch(xb, zb, valid, fake, delta,
                                                   critic_x_model, critic_x, optimizer))
            cz_loss.append(critic_z_train_on_batch(xb, zb, valid, fake, delta,
                                                   critic_z_model, critic_z, optimizer))
        critic_x.trainable = False; critic_z.trainable = False
        generator.trainable = True;  encoder.trainable = True
        g_loss.append(enc_gen_train_on_batch(xb, zb, valid, encoder_generator_model, optimizer))
    print('Epoch {}/{}, [Dx {}] [Dz {}] [G {}]'.format(
        epoch, epochs, np.mean(np.array(cx_loss), axis=0),
        np.mean(np.array(cz_loss), axis=0), np.mean(np.array(g_loss), axis=0)))

## 4) 저장

In [ ]:
critic_x_model.save_weights(os.path.join(ckpt_dir, 'critic_x_model.h5'), save_format='h5')
critic_z_model.save_weights(os.path.join(ckpt_dir, 'critic_z_model.h5'), save_format='h5')
encoder_generator_model.save_weights(os.path.join(ckpt_dir, 'encoder_generator_model.h5'), save_format='h5')
print("saved checkpoints ->", ckpt_dir)
print("\n[실험 방법] CELL_IDS=[1000] 로 단일학습, [1000,1001,1002] 로 합침학습을")
print("각각 돌려 03_test 결과를 비교하면 '정상 데이터 양 -> 성능' 효과를 볼 수 있다.")